# Modul 3 · Teknik Klasifikasi dan Prediksi
**Pelatihan Machine Learning & Data Analysis**

Dua jenis pertanyaan bisnis → dua teknik ML yang berbeda:

1. **KLASIFIKASI** — memprediksi *kategori / label*
   - "Konsumen kredit ini nanti **Lancar** atau **Macet**?"
   - "Transaksi ini **wajar** atau **mencurigakan**?"
2. **REGRESI / FORECASTING** — memprediksi *angka*
   - "Berapa **unit** motor yang terjual bulan depan?"
   - "Berapa kebutuhan **stok** sparepart kuartal depan?"

Isi notebook ini:
- **Bagian A** — Klasifikasi risiko kredit konsumen (Random Forest)
- **Bagian B** — Forecasting penjualan 6 bulan ke depan

> ⚠️ Prasyarat: notebook `00_generate_dataset.ipynb` sudah dijalankan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

DIR = Path.cwd().parent

## BAGIAN A : KLASIFIKASI - PREDIKSI RISIKO KREDIT (LANCAR / MACET)

In [ ]:
kredit = pd.read_csv(DIR / "dataset" / "data_kredit_konsumen.csv")
print(f"\nData historis: {len(kredit)} konsumen")
print(kredit["status_pembayaran"].value_counts().to_string())

# Fitur (X) = profil konsumen | Target (y) = status pembayaran
fitur = ["usia", "penghasilan_juta", "dp_persen",
         "tenor_bulan", "jumlah_telat_sebelumnya"]
X = kredit[fitur]
y = kredit["status_pembayaran"]

# Bagi data latih/uji secara acak (80% belajar : 20% ujian)
X_latih, X_uji, y_latih, y_uji = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Random Forest = "ratusan pohon keputusan yang voting bersama"
model_klas = RandomForestClassifier(n_estimators=200, random_state=42)
model_klas.fit(X_latih, y_latih)

prediksi = model_klas.predict(X_uji)
akurasi = accuracy_score(y_uji, prediksi)

print(f"\nAkurasi pada data uji : {akurasi:.1%} "
      f"({(prediksi == y_uji).sum()} benar dari {len(y_uji)} konsumen)")

# Confusion matrix = rincian benar/salahnya prediksi
cm = confusion_matrix(y_uji, prediksi, labels=["Lancar", "Macet"])
print("\nConfusion Matrix (baris = kenyataan, kolom = prediksi):")
print(pd.DataFrame(cm,
                   index=["Aktual Lancar", "Aktual Macet"],
                   columns=["Prediksi Lancar", "Prediksi Macet"]).to_string())
print("\nPenting dipahami: 'Aktual Macet tapi Diprediksi Lancar' adalah "
      "kesalahan\npaling MAHAL bagi bisnis (kredit berisiko lolos persetujuan).")

print("\nLaporan lengkap per kelas:")
print(classification_report(y_uji, prediksi))

### Pakai model untuk konsumen BARU (inilah nilai bisnisnya!)

In [ ]:
calon_baru = pd.DataFrame([
    {"usia": 24, "penghasilan_juta": 4.0, "dp_persen": 10,
     "tenor_bulan": 36, "jumlah_telat_sebelumnya": 3},
    {"usia": 38, "penghasilan_juta": 12.0, "dp_persen": 30,
     "tenor_bulan": 12, "jumlah_telat_sebelumnya": 0},
])
label = model_klas.predict(calon_baru)
peluang = model_klas.predict_proba(calon_baru)
kolom_macet = list(model_klas.classes_).index("Macet")

print("SIMULASI: dua calon konsumen baru mengajukan kredit")
for i in range(len(calon_baru)):
    print(f"  Calon #{i + 1} {calon_baru.iloc[i].to_dict()}")
    print(f"     -> Prediksi: {label[i]} "
          f"(peluang macet {peluang[i, kolom_macet]:.0%})")

## BAGIAN B : PREDIKSI ANGKA - FORECASTING PENJUALAN 6 BULAN KE DEPAN

In [ ]:
jual = pd.read_csv(DIR / "dataset" / "data_penjualan_motor.csv")
jual["urutan_bulan"] = range(1, len(jual) + 1)

# Trik penting deret waktu: ubah "bulan" menjadi pola melingkar (sin & cos)
# supaya model paham bahwa Desember dan Januari itu berdekatan.
jual["musim_sin"] = np.sin(2 * np.pi * jual["no_bulan"] / 12)
jual["musim_cos"] = np.cos(2 * np.pi * jual["no_bulan"] / 12)

fitur_fc = ["urutan_bulan", "musim_sin", "musim_cos", "ada_promo"]
model_fc = RandomForestRegressor(n_estimators=300, random_state=42)
model_fc.fit(jual[fitur_fc], jual["unit_terjual"])

# Susun 6 bulan ke depan (Jun-Nov 2026), skenario: promo di Jul & Nov
masa_depan = pd.DataFrame({
    "bulan": pd.date_range("2026-06-01", periods=6, freq="MS").strftime("%Y-%m"),
    "urutan_bulan": range(len(jual) + 1, len(jual) + 7),
    "no_bulan": list(range(6, 12)),
    "ada_promo": [0, 1, 0, 0, 0, 1],
})
masa_depan["musim_sin"] = np.sin(2 * np.pi * masa_depan["no_bulan"] / 12)
masa_depan["musim_cos"] = np.cos(2 * np.pi * masa_depan["no_bulan"] / 12)
masa_depan["prediksi_unit"] = (
    model_fc.predict(masa_depan[fitur_fc]).round().astype(int))

print("\nPrediksi penjualan (skenario promo di Juli & November):")
print(masa_depan[["bulan", "ada_promo", "prediksi_unit"]]
      .to_string(index=False))

total_fc = masa_depan["prediksi_unit"].sum()
print(f"\nPerkiraan total 6 bulan : {total_fc:,} unit "
      f"-> dasar perencanaan produksi & stok.")

### Visualisasi riwayat + ramalan

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(jual["urutan_bulan"], jual["unit_terjual"],
         color="#444444", label="Riwayat penjualan")
plt.plot(masa_depan["urutan_bulan"], masa_depan["prediksi_unit"],
         "o--", color="#E60012", label="Forecast 6 bulan ke depan")
plt.axvline(len(jual) + 0.5, color="gray", linestyle=":")
plt.title("Forecasting Penjualan: Riwayat vs Prediksi Masa Depan")
plt.xlabel("Bulan ke-")
plt.ylabel("Unit terjual")
plt.legend()
plt.tight_layout()
file_grafik = DIR / "output" / "modul3_forecast_penjualan.png"
plt.savefig(file_grafik, dpi=120)
plt.show()
print(f"Grafik disimpan ke: {file_grafik}")

print("""
--------------------------------------------------------------------------------
KESIMPULAN MODUL 3
--------------------------------------------------------------------------------
1. Pertanyaan "kategori apa?"  -> KLASIFIKASI  (contoh: Lancar/Macet).
2. Pertanyaan "berapa angkanya?" -> REGRESI/FORECASTING (contoh: unit terjual).
3. Confusion matrix membantu melihat JENIS kesalahan, bukan cuma akurasi total,
   karena tiap jenis kesalahan punya biaya bisnis yang berbeda.
4. Hasil forecast langsung bisa dipakai: rencana produksi, stok, & target sales.
--------------------------------------------------------------------------------
""")

---
### 💡 Latihan mandiri
1. Ubah profil dua calon konsumen baru (usia, penghasilan, DP, tenor, riwayat telat) dan amati perubahan peluang macetnya.
2. Di Bagian B, ubah skenario promo (mis. promo hanya di November) lalu bandingkan total forecast 6 bulannya.